# Graph-based Knowledge Tracing (GKT) - Kaggle Runner

Notebook clone repo [Knowledge-Tracing](https://github.com/linhnnh688/Knowledge-Tracing) lam codebase, chay train + evaluate GKT tren ASSISTments.

**Pipeline:**
1. Clone repo, cai dependency
2. EDA nhanh
3. Train GKT (Dense, Transition, MHA, VAE, PAM)
4. Train DKT baseline
5. danh gia AUC / Accuracy
6. Visualization

## 0. Setup - Clone repo & Install

In [ ]:
!git clone https://github.com/linhnnh688/Knowledge-Tracing.git /kaggle/working/Knowledge-Tracing
%cd /kaggle/working/Knowledge-Tracing/GKT

In [ ]:
!pip install numpy pandas scipy scikit-learn torch -q

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/Knowledge-Tracing/GKT')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Exploratory Data Analysis

In [ ]:
DATA_DIR = '/kaggle/working/Knowledge-Tracing/GKT/data'

df_small = pd.read_csv(f'{DATA_DIR}/assistment_test15.csv')
df_full = pd.read_csv(f'{DATA_DIR}/skill_builder_data.csv')

print('=== assistment_test15.csv ===')
print(f'Shape: {df_small.shape}')
print(f'Users: {df_small["user_id"].nunique()}')
print(f'Skills: {df_small["skill_id"].nunique()}')
print(f'Correct ratio: {df_small["correct"].mean():.3f}')
print()
print('=== skill_builder_data.csv ===')
print(f'Shape: {df_full.shape}')
print(f'Users: {df_full["user_id"].nunique()}')
print(f'Skills: {df_full["skill_id"].nunique()}')
print(f'Correct ratio: {df_full["correct"].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, df) in zip(axes, [('assistment_test15', df_small), ('skill_builder', df_full)]):
    user_counts = df.groupby('user_id').size()
    ax.hist(user_counts, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'{name}: Sequence Length Distribution')
    ax.set_xlabel('Interactions per user')
    ax.set_ylabel('Count')
    ax.axvline(user_counts.median(), color='red', linestyle='--', label=f'Median={user_counts.median():.0f}')
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Training Function

In [ ]:
import time, gc, datetime
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.autograd import Variable
from models import GKT, MultiHeadAttention, VAE, DKT
from metrics import KTLoss, VAELoss
from processing import load_dataset
import random

def run_experiment(
    data_file, model_type='GKT', graph_type='Dense',
    epochs=30, batch_size=128, lr=0.001,
    hid_dim=32, emb_dim=32, attn_dim=32,
    edge_types=2, dropout=0.0,
    train_ratio=0.6, val_ratio=0.2,
    seed=42, device=None
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

    data_path = f'{DATA_DIR}/{data_file}'
    use_cuda = torch.cuda.is_available()

    concept_num, graph, train_loader, valid_loader, test_loader = load_dataset(
        data_path, batch_size, graph_type,
        train_ratio=train_ratio, val_ratio=val_ratio,
        shuffle=True, model_type=model_type, use_cuda=use_cuda
    )

    res_len = 2
    graph_model = None

    if model_type == 'GKT':
        if graph_type == 'MHA':
            graph_model = MultiHeadAttention(edge_types, concept_num, emb_dim, attn_dim, dropout=dropout)
        elif graph_type == 'VAE':
            graph_model = VAE(emb_dim, 32, edge_types, 32, 32, concept_num,
                             edge_type_num=edge_types, tau=0.5, factor=True, dropout=dropout, bias=True)
        if use_cuda and graph_type in ['MHA', 'VAE']:
            graph_model = graph_model.cuda()
        model = GKT(concept_num, hid_dim, emb_dim, edge_types, graph_type,
                     graph=graph, graph_model=graph_model,
                     dropout=dropout, bias=True, has_cuda=use_cuda)
    elif model_type == 'DKT':
        model = DKT(res_len * concept_num, emb_dim, concept_num, dropout=dropout, bias=True)
    else:
        raise ValueError(f'Unknown model_type: {model_type}')

    kt_loss = KTLoss()
    vae_loss = None
    if model_type == 'GKT' and graph_type == 'VAE':
        vae_loss = VAELoss(concept_num, edge_type_num=edge_types)

    if use_cuda:
        model = model.cuda()
        if vae_loss is not None:
            vae_loss = vae_loss.cuda()

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=200, gamma=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': [],
               'train_acc': [], 'val_acc': []}
    best_val_loss = np.inf
    best_state = None

    for epoch in range(epochs):
        model.train()
        if hasattr(model, 'graph_model') and model.graph_model is not None:
            model.graph_model.train()

        ep_train_loss, ep_train_auc, ep_train_acc = [], [], []
        for features, questions, answers in train_loader:
            features, questions, answers = features.to(device), questions.to(device), answers.to(device)
            if model_type == 'GKT':
                pred_res, _, _, _ = model(features, questions)
            else:
                pred_res = model(features, questions)

            loss_kt, auc, acc = kt_loss(pred_res, answers)
            loss = loss_kt
            if model_type == 'GKT' and graph_type == 'VAE':
                _, ec_list, rec_list, z_prob_list = model(features, questions)
                loss_vae = vae_loss(ec_list, rec_list, z_prob_list)
                loss = loss_kt + loss_vae

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            ep_train_loss.append(loss_kt.item())
            if auc != -1:
                ep_train_auc.append(auc)
                ep_train_acc.append(acc)

        model.eval()
        if hasattr(model, 'graph_model') and model.graph_model is not None:
            model.graph_model.eval()

        ep_val_loss, ep_val_auc, ep_val_acc = [], [], []
        with torch.no_grad():
            for features, questions, answers in valid_loader:
                features, questions, answers = features.to(device), questions.to(device), answers.to(device)
                if model_type == 'GKT':
                    pred_res, _, _, _ = model(features, questions)
                else:
                    pred_res = model(features, questions)
                loss_kt, auc, acc = kt_loss(pred_res, answers)
                ep_val_loss.append(loss_kt.item())
                if auc != -1:
                    ep_val_auc.append(auc)
                    ep_val_acc.append(acc)

        mean_tl = np.mean(ep_train_loss)
        mean_vl = np.mean(ep_val_loss)
        mean_ta = np.mean(ep_train_auc) if ep_train_auc else 0
        mean_va = np.mean(ep_val_auc) if ep_val_auc else 0
        mean_tacc = np.mean(ep_train_acc) if ep_train_acc else 0
        mean_vacc = np.mean(ep_val_acc) if ep_val_acc else 0

        history['train_loss'].append(mean_tl)
        history['val_loss'].append(mean_vl)
        history['train_auc'].append(mean_ta)
        history['val_auc'].append(mean_va)
        history['train_acc'].append(mean_tacc)
        history['val_acc'].append(mean_vacc)

        if mean_vl < best_val_loss:
            best_val_loss = mean_vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | Train Loss={mean_tl:.4f} AUC={mean_ta:.4f} Acc={mean_tacc:.4f} | Val Loss={mean_vl:.4f} AUC={mean_va:.4f} Acc={mean_vacc:.4f}')

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    test_loss, test_auc, test_acc = [], [], []
    with torch.no_grad():
        for features, questions, answers in test_loader:
            features, questions, answers = features.to(device), questions.to(device), answers.to(device)
            if model_type == 'GKT':
                pred_res, _, _, _ = model(features, questions)
            else:
                pred_res = model(features, questions)
            loss_kt, auc, acc = kt_loss(pred_res, answers)
            test_loss.append(loss_kt.item())
            if auc != -1:
                test_auc.append(auc)
                test_acc.append(acc)

    results = {
        'model': model_type,
        'graph_type': graph_type if model_type == 'GKT' else 'N/A',
        'test_loss': np.mean(test_loss),
        'test_auc': np.mean(test_auc) if test_auc else 0,
        'test_acc': np.mean(test_acc) if test_acc else 0,
        'history': history,
        'concept_num': concept_num
    }
    print(f'\n>>> Test: Loss={results["test_loss"]:.4f} | AUC={results["test_auc"]:.4f} | Acc={results["test_acc"]:.4f}')
    return results

## 3. DKT Baseline

In [ ]:
print('='*60)
print('DKT Baseline on assistment_test15')
print('='*60)
results_dkt = run_experiment(
    data_file='assistment_test15.csv',
    model_type='DKT',
    epochs=30, batch_size=128, lr=0.001,
    hid_dim=32, emb_dim=32, seed=42
)

## 4. GKT Experiments (Dense, Transition, MHA, VAE, PAM)

In [ ]:
graph_types = ['Dense', 'Transition', 'MHA', 'VAE', 'PAM']
all_results = [results_dkt]

for gt in graph_types:
    print('\n' + '='*60)
    print(f'GKT with graph_type={gt} on assistment_test15')
    print('='*60)
    try:
        res = run_experiment(
            data_file='assistment_test15.csv',
            model_type='GKT', graph_type=gt,
            epochs=30, batch_size=128, lr=0.001,
            hid_dim=32, emb_dim=32, attn_dim=32,
            edge_types=2, seed=42
        )
        all_results.append(res)
    except Exception as e:
        print(f'  [ERROR] {gt}: {e}')

## 5. Bang tong hop ket qua

In [ ]:
summary = []
for r in all_results:
    summary.append({
        'Model': r['model'],
        'Graph Type': r['graph_type'],
        'Test Loss': f'{r["test_loss"]:.4f}',
        'Test AUC': f'{r["test_auc"]:.4f}',
        'Test Acc': f'{r["test_acc"]:.4f}'
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(all_results)))

for i, r in enumerate(all_results):
    label = r['model'] if r['graph_type'] == 'N/A' else f"GKT-{r['graph_type']}"
    h = r['history']
    axes[0].plot(h['val_loss'], label=label, color=colors[i])
    axes[1].plot(h['val_auc'], label=label, color=colors[i])

axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('NLL Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('GKT vs DKT - assistment_test15.csv', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
model_names = [r['model'] if r['graph_type'] == 'N/A' else f"GKT-{r['graph_type']}" for r in all_results]
aucs = [r['test_auc'] for r in all_results]
accs = [r['test_acc'] for r in all_results]
x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, aucs, width, label='AUC', color='steelblue')
bars2 = ax.bar(x + width/2, accs, width, label='Accuracy', color='coral')
ax.set_ylabel('Score')
ax.set_title('Test AUC & Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Multi-seed evaluation (mean +/- std)

Chay 3 seed de bao cao mean +/- std theo yeu cau do an.

In [ ]:
seeds = [42, 123, 456]
multi_seed_results = {}

for model_cfg in [
    {'model_type': 'DKT', 'graph_type': 'N/A'},
    {'model_type': 'GKT', 'graph_type': 'Dense'},
    {'model_type': 'GKT', 'graph_type': 'Transition'},
]:
    name = model_cfg['model_type'] if model_cfg['graph_type'] == 'N/A' else f"GKT-{model_cfg['graph_type']}"
    aucs, accs = [], []
    for s in seeds:
        print(f'\n--- {name} | seed={s} ---')
        r = run_experiment(
            data_file='assistment_test15.csv',
            model_type=model_cfg['model_type'],
            graph_type=model_cfg.get('graph_type', 'Dense'),
            epochs=20, batch_size=128, lr=0.001,
            hid_dim=32, emb_dim=32, seed=s
        )
        aucs.append(r['test_auc'])
        accs.append(r['test_acc'])
    multi_seed_results[name] = {'auc_mean': np.mean(aucs), 'auc_std': np.std(aucs),
                                  'acc_mean': np.mean(accs), 'acc_std': np.std(accs)}

print('\n' + '='*60)
print('Multi-Seed Results (3 seeds)')
print('='*60)
for name, v in multi_seed_results.items():
    print(f'{name:20s} | AUC: {v["auc_mean"]:.4f} +/- {v["auc_std"]:.4f} | Acc: {v["acc_mean"]:.4f} +/- {v["acc_std"]:.4f}')

## 8. Luu ket qua

In [ ]:
df_summary.to_csv('/kaggle/working/gkt_results.csv', index=False)
print('Saved: /kaggle/working/gkt_results.csv')
df_summary